In [2]:
# Imports
#  /home/bradachi/Documentos/GitHub/NLP_teste/NLP_ENV/bin/python -m pip install 
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
import string
from textblob import TextBlob
from LeIA import SentimentIntensityAnalyzer
from datetime import datetime
import re
#import requests
#import json
#import pprint
#import lxml
#import time
#from wordcloud import WordCloud, STOPWORDS, ImageColorGenerator
#import matplotlib.pyplot as plt 
# from bertopic import BERTopic
#from PIL import Image
#import numpy as np
#from statsmodels.distributions.empirical_distribution import ECDF
#import matplotlib.ticker as mtick
# from selenium import webdriver
# from webdriver_manager.firefox import GeckoDriverManager 
# from selenium.webdriver.firefox.service import Service 
# from selenium.webdriver.common.action_chains import ActionChains
# from selenium.webdriver.common.keys import Keys
# from selenium.common.exceptions import NoSuchElementException


[nltk_data] Downloading package punkt to /home/bradachi/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/bradachi/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/bradachi/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
produtos = pd.read_csv("../data/general_products/produtos.csv")
produtos

,id,date_created,catalog_product_id,domain_id,name,main_features,quality_type,priority,type,keywords,description
0,MLB46241561,2025-02-21T13:14:52Z,MLB46241561,MLB-REFRIGERATOR_AND_FREEZER_SENSORS,Sensores Temperatura/evaporador Split Electrol...,[],COMPLETE,MEDIUM,PRODUCT,Sensores Temperatura/evaporador Split Electrol...,NaN
1,MLB39940858,2024-08-27T21:56:42Z,MLB39940858,MLB-REFRIGERATOR_AND_FREEZER_SENSORS,2 Sensores De Temperatura Refrigerador Crd36,[],COMPLETE,MEDIUM,PRODUCT,2 Sensores De Temperatura Refrigerador Crd36,NaN
2,MLB48776606,2025-04-19T02:23:06Z,MLB48776606,MLB-REFRIGERATOR_AND_FREEZER_SENSORS,Termofusivel 105c + 2 Sensores Temperatura Degelo,[],PRIMITIVE,MEDIUM,PRODUCT,Termofusivel 105c + 2 Sensores Temperatura Degelo,NaN
3,MLB59018447,2025-10-05T02:13:07Z,MLB59018447,MLB-REFRIGERATOR_AND_FREEZER_SENSORS,Kit 2 Sensores 5k Geladeira Electrolux,[],COMPLETE,MEDIUM,PRODUCT,Kit 2 Sensores 5k Geladeira Electrolux,NaN
4,MLB44743999,2024-12-13T15:48:20Z,MLB44743999,MLB-REFRIGERATOR_AND_FREEZER_SENSORS,Termostato Bimetal +sensores Temp. P/ Cervejei...,[],COMPLETE,MEDIUM,PRODUCT,Termostato Bimetal +sensores Temp. P/ Cervejei...,NaN
...,...,...,...,...,...,...,...,...,...,...,...
23039,MLB38765729,2024-07-31T01:39:07Z,MLB38765729,MLB-DIMMERS,Interruptor inteligente Wi-Fi para ventilador ...,[],COMPLETE,MEDIUM,PRODUCT,Interruptor inteligente Wi-Fi para ventilador ...,NaN
23040,MLB45805005,2025-01-30T19:51:00Z,MLB45805005,MLB-DIMMERS,Interruptor Inteligente Zigbee Novadigital 2 T...,[],COMPLETE,MEDIUM,PRODUCT,Interruptor Inteligente Zigbee Novadigital 2 T...,NaN
23041,MLB47688342,2025-04-04T17:32:34Z,MLB47688342,MLB-DIMMERS,Interruptor Inteligente Zigbee Novadigital 1 T...,[],COMPLETE,MEDIUM,PRODUCT,Interruptor Inteligente Zigbee Novadigital 1 T...,NaN
23042,MLB39454456,2024-08-22T19:56:27Z,MLB39454456,MLB-DIMMERS,Interruptor Inteligente Wi-fi Com Função Dimme...,[],COMPLETE,MEDIUM,PRODUCT,Interruptor Inteligente Wi-fi Com Função Dimme...,NaN


In [4]:
avaliacoes = pd.read_csv("../data/general_products/avaliacoes.csv")
avaliacoes

,texto,nota,id,data
0,Excelente produto!!.,5,MLB46241561,09 out. 2025
1,Muito bom.,5,MLB48090512,15 out. 2025
2,Muito bom produto.,5,MLB48090512,16 out. 2025
3,Produto bom. Sem retorno.,5,MLB48090512,20 nov. 2025
4,Produto confiável e original. Eu mesmo arrumei...,5,MLB29176649,27 out. 2025
...,...,...,...,...
9648,Atendeu super bem para controles de ventilador...,5,MLB50911897,28 nov. 2025
9649,Produto excelente e facil instalacao.,5,MLB50911897,01 dez. 2025
9650,"Simples, rápido e eficiente. Produto muito bom...",5,MLB50911897,25 out. 2025
9651,É elogiado por sua facilidade de configuração ...,5,MLB50911897,20 nov. 2025


In [5]:
perguntas = pd.read_csv("../data/general_products/perguntas.csv")
perguntas

,pergunta,resposta,id,data
0,Minha cervejeira é VN28AF Termostato sollatek ...,"Boa tarde, seu termostato saiu de linha o subs...",MLB44743999,27/11/2025
1,Esse termostato tem os 4 pinos?,"Boa tarde, são 6 pinos porem os que tem função...",MLB44743999,26/11/2025
2,Teria para o modelo VF56DB 220V,"Boa tarde, seu modelo usa termostato sollatek,...",MLB44743999,26/06/2025
3,Ok ele e 220v,O termostato do anúncio é bivolt.,MLB44743999,15/05/2025
4,Bom dia ele serve para o modelo vb15 metal frio,"Bom dia, sim temos programações para expositor...",MLB44743999,16/05/2025
...,...,...,...,...
28994,Olá é luz quente,"Olá Tudo bem? Não, a luz é branco neutro 4000k...",MLB46936892,02/09/2025
28995,Precisso da nota fiscal de este produto,Olá boa tarde! Tudo bem? Claro! Pode nos envia...,MLB46936892,23/06/2025
28996,"Muito obrigado pela atenção, E27 que preciso. ...",Deu certo o Sr. conseguiu utilizar?,MLB46936892,29/05/2025
28997,Comprei chegou mas a rosca não é E27 é menor,Boa noite! Tudo bem? Mas esse modelo não temos...,MLB46936892,28/05/2025


In [6]:
contextos = pd.read_csv("../data/general_products/contexto.csv")
contextos

,Context_Id,Context,value_id,value_name,id
0,BRAND,Marca,188.0,Electrolux,MLB46241561
1,MODEL,Modelo,47713338.0,YI09F YI09R YI12F YI12R,MLB46241561
2,SENSOR_TYPE,Tipo de sensor,39861884.0,Temperatura,MLB46241561
3,WEIGHT,Peso,19173692.0,0.2 g,MLB46241561
4,BRAND,Marca,17452295.0,GBMak,MLB39940858
...,...,...,...,...,...
299579,IS_PROGRAMMABLE,É programável,242085.0,Sim,MLB44773654
299580,WITH_WI_FI,Com Wi-Fi,242084.0,Não,MLB44773654
299581,ANATEL_HOMOLOGATION_NUMBER,Homologação Anatel Nº,39981355.0,104162111765,MLB44773654
299582,MANUFACTURER,Fabricante,2449062.0,Novadigital,MLB44773654


In [7]:
wifiObjects = contextos[contextos["Context_Id"] == "WITH_WI_FI"]
wifiObjects2 = contextos[contextos["Context_Id"] == "WITH_WI_FI_CONNECTION"]
wifiObjects3 = contextos[contextos["Context_Id"] == "IS_COMPATIBLE_WITH_WI_FI_MODULE"]
wifiObjects4 = contextos[contextos["Context_Id"] == "WITH_WIFI"]
bltObjects = contextos[contextos["Context_Id"] == "WITH_BLUETOOTH"]
mobNet = contextos[contextos["Context_Id"] == "WITH_MOBILE_NETWORK"]
nfc = contextos[contextos["Context_Id"] == "WITH_NFC"]
wl = contextos[contextos["Context_Id"] == "IS_WIRELESS"]

mobConn = contextos[contextos["Context_Id"] == "MOBILE_CONNECTIONS"]
connTypesObjects = contextos[contextos["Context_Id"] == "CONNECTION_TYPES"]
wireless = contextos[contextos["Context_Id"] == "WIRELESS_STANDARDS"]
connObjects = contextos[contextos["Context_Id"] == "CONNECTIVITY"]
mobNetType = contextos[contextos["Context_Id"] == "MOBILE_NETWORK"]
comp1 = contextos[contextos["Context_Id"] == "COMPATIBLE_VIRTUAL_ASSISTANTS"]
comp2 = contextos[contextos["Context_Id"] == "COMPATIBLE_SMART_APPS"]


smartObjectsID = pd.concat([wifiObjects, wifiObjects2, 
                            wifiObjects3, wifiObjects4, 
                            bltObjects, mobNet, nfc, wl])

smartObjectsID = smartObjectsID[smartObjectsID["value_name"] == "Sim"]

smartObjectsID = pd.concat([smartObjectsID, connTypesObjects, 
                            connObjects, wireless, mobNetType,
                            comp1, comp2, mobConn])

smartObjectsID = smartObjectsID["id"].drop_duplicates().reset_index(drop=True)
smartObjectsID = list(smartObjectsID)
smartObjectsID

['MLB36333923',
 'MLB36139109',
 'MLB21345002',
 'MLB38703841',
 'MLB36398531',
 'MLB54448099',
 'MLB48677687',
 'MLB19552926',
 'MLB27839322',
 'MLB28002506',
 'MLB56994543',
 'MLB29568434',
 'MLB26446377',
 'MLB38750127',
 'MLB54236783',
 'MLB41626835',
 'MLB24455199',
 'MLB57356995',
 'MLB27985244',
 'MLB28682642',
 'MLB43616708',
 'MLB27769043',
 'MLB34345595',
 'MLB28754492',
 'MLB61666608',
 'MLB40002972',
 'MLB25735509',
 'MLB41799760',
 'MLB39171159',
 'MLB28037061',
 'MLB28834431',
 'MLB51925280',
 'MLB35985980',
 'MLB23222747',
 'MLB45867674',
 'MLB36956947',
 'MLB27238649',
 'MLB20049271',
 'MLB20545758',
 'MLB29417181',
 'MLB26085789',
 'MLB42639627',
 'MLB39983380',
 'MLB20662835',
 'MLB26051587',
 'MLB52139663',
 'MLB22145672',
 'MLB24198079',
 'MLB37791461',
 'MLB24228530',
 'MLB58376900',
 'MLB36411103',
 'MLB36034704',
 'MLB23942605',
 'MLB44845954',
 'MLB22031377',
 'MLB40543976',
 'MLB48710929',
 'MLB60176792',
 'MLB48848053',
 'MLB38347793',
 'MLB38264333',
 'MLB385

In [8]:
smartObjectsReviews = ( avaliacoes[avaliacoes["id"].isin(smartObjectsID)]
                       .drop_duplicates()
                       .reset_index(drop=True)
                       .dropna() )
smartObjectsReviews.to_csv('../data/smart_objects/smart_objects_reviews.csv', index=False)


In [9]:
smartObjectsQuestions = (perguntas[perguntas["id"].isin(smartObjectsID)]
                         .drop_duplicates()
                         .reset_index(drop=True))
smartObjectsQuestions.to_csv('../data/smart_objects/smart_objects_questions.csv', index=False)

In [10]:
smartObjects = produtos[produtos["id"].isin(smartObjectsID)].drop_duplicates().reset_index(drop=True) 
smartObjects.to_csv('../data/smart_objects/smart_objects_products.csv', index=False)

In [11]:
smartObjectsContexts = contextos[contextos["id"].isin(smartObjectsID)].drop_duplicates().reset_index(drop=True) 
smartObjectsContexts.to_csv('../data/smart_objects/smart_objects_contexts.csv', index=False)

# Gráfico de % pos neg

In [12]:
# função de remoção de stopwords
def removerstopwords(texto):
    # deixando texto em minúsculo
    texto = texto.lower()

    # removendo pontuações do texto
    pontuacao = str.maketrans('', '', string.punctuation)
    texto = texto.translate(pontuacao)

    # removendo stopwords do texto
    palavras = word_tokenize(texto, language='portuguese')
    stopwaords = set(stopwords.words('portuguese'))
    palavrasfiltradas = [palavra for palavra in palavras if palavra.lower() not in stopwaords]

    # removendo espaços vazios restantes dos tokens
    listasemvazios = list(filter(None, palavrasfiltradas))

    return listasemvazios

def analisesentimentosBlob(texto):
  # Cria o objeto TextBlob
  blob = TextBlob(texto)

  # Pega o sentimento
  # se > 0.1 é positivo, se 0 é neutro e se < -0.1 é negativo
  sentimento = blob.sentiment
  return sentimento.polarity

def analisesentimentosLeIA(texto):
  # Cria o objeto LeIA
  s = SentimentIntensityAnalyzer()


  # Pega o sentimento
  # se > 0.05 é positivo, entre é neutro e se < -0.05 é negativo
  sentimento = s.polarity_scores(texto)
  return sentimento["compound"]

def tiposentimento(x):
  if x > 0.05:
    return 'Positivo'
  if x < -0.05:
    return 'Negativo'
  else:
    return 'Neutro'

def calcula_porcentagem_sentimento(df:pd.DataFrame)->dict:
    res = dict()
    for domain in df['domain_id'].unique():
        df_domain = df[df['domain_id'] == domain]
        pos = df_domain[df_domain['tiposentimento'] == 'Positivo']
        neg = df_domain[df_domain['tiposentimento'] == 'Negativo']
        tot = len(pos)+len(neg)
        res[domain] = {'pos': len(pos)/tot, "neg": len(neg)/tot}
        # res[domain] = {'pos': len(pos), "neg": len(neg)}
    return res

In [13]:
smartObjectsReviews['sentimento'] = smartObjectsReviews['texto'].apply(analisesentimentosLeIA)
smartObjectsReviews['tiposentimento'] = smartObjectsReviews['sentimento'].apply(tiposentimento)
smartObjectsReviews = pd.merge(smartObjectsReviews, smartObjects[['id', 'domain_id']], on='id', how='left').reset_index(drop=True)
smartObjectsReviews.to_csv('../data/smart_objects/smart_objects_reviews.csv', index=False)

In [14]:
smartObjectsQuestions['sentimento_pergunta'] = smartObjectsQuestions['pergunta'].apply(analisesentimentosLeIA)
smartObjectsQuestions['tiposentimento_pergunta'] = smartObjectsQuestions['sentimento_pergunta'].apply(tiposentimento)
smartObjectsQuestions = pd.merge(smartObjectsQuestions, smartObjects[['id', 'domain_id']], on='id', how='left').reset_index(drop=True)

smartObjectsQuestions['sentimento_resposta'] = smartObjectsQuestions['resposta'].apply(analisesentimentosLeIA)
smartObjectsQuestions['tiposentimento_resposta'] = smartObjectsQuestions['sentimento_resposta'].apply(tiposentimento)
smartObjectsQuestions = pd.merge(smartObjectsQuestions, smartObjects[['id', 'domain_id']], on='id', how='left').reset_index(drop=True)


smartObjectsQuestions.to_csv('../data/smart_objects/smart_objects_questions.csv', index=False)

In [54]:
# ajustando datas

def ajusta_data(data):
    meses = {
        'jan': '01', 'fev': '02', 'mar': '03', 'abr': '04',
        'mai': '05', 'jun': '06', 'jul': '07', 'ago': '08',
        'set': '09', 'out': '10', 'nov': '11', 'dez': '12'
    }
    
    if re.fullmatch("[0-9]{2}/[0-9]{2}/[0-9]{4}", data):
        data = datetime.strptime(data, "%d/%m/%Y").isoformat()
    else:
        padrao = re.compile(r'\b(' + '|'.join(meses.keys()) + r')(\.)?\b') # encontra substring
        data = padrao.sub(lambda x: meses[x.group().lower()], data) # altera substring

        data = datetime.strptime(data, "%d %m. %Y").isoformat()

    return data


In [55]:
smartObjectsQuestions['data'] = smartObjectsQuestions['data'].apply(ajusta_data)
smartObjectsQuestions.to_csv('../data/smart_objects/smart_objects_questions.csv', index=False)


In [53]:
smartObjectsReviews = pd.read_csv("../data/smart_objects/smart_objects_reviews.csv")
smartObjectsQuestions = pd.read_csv("../data/smart_objects/smart_objects_questions.csv")
avaliacoes = pd.read_csv("../data/general_products/avaliacoes.csv")
perguntas = pd.read_csv("../data/general_products/perguntas.csv")


In [50]:
smartObjectsReviews['data'] = smartObjectsReviews['data'].apply(ajusta_data)
avaliacoes['data'] = avaliacoes['data'].apply(ajusta_data)
perguntas['data'] = perguntas['data'].apply(ajusta_data)

smartObjectsQuestions.to_csv('../data/smart_objects/smart_objects_questions.csv', index=False)
smartObjectsReviews.to_csv('../data/smart_objects/smart_objects_reviews.csv', index=False)
avaliacoes.to_csv('../data/general_products/avaliacoes.csv', index=False)
perguntas.to_csv('../data/general_products/perguntas.csv', index=False)



debug 13 dez. 2024
DEBUG 13 12. 2024
debug 16 jan. 2025
DEBUG 16 01. 2025
debug 29 mar. 2025
DEBUG 29 03. 2025
debug 08 mai. 2025
DEBUG 08 05. 2025
debug 24 jul. 2025
DEBUG 24 07. 2025
debug 02 set. 2025
DEBUG 02 09. 2025
debug 22 set. 2025
DEBUG 22 09. 2025
debug 29 set. 2025
DEBUG 29 09. 2025
debug 08 out. 2025
DEBUG 08 10. 2025
debug 20 nov. 2025
DEBUG 20 11. 2025
debug 08 out. 2024
DEBUG 08 10. 2024
debug 20 nov. 2025
DEBUG 20 11. 2025
debug 22 set. 2025
DEBUG 22 09. 2025
debug 05 out. 2025
DEBUG 05 10. 2025
debug 23 out. 2025
DEBUG 23 10. 2025
debug 24 out. 2025
DEBUG 24 10. 2025
debug 22 nov. 2025
DEBUG 22 11. 2025
debug 27 nov. 2025
DEBUG 27 11. 2025
debug 20 nov. 2025
DEBUG 20 11. 2025
debug 11 out. 2025
DEBUG 11 10. 2025
debug 02 nov. 2025
DEBUG 02 11. 2025
debug 06 nov. 2025
DEBUG 06 11. 2025
debug 12 nov. 2025
DEBUG 12 11. 2025
debug 14 nov. 2025
DEBUG 14 11. 2025
debug 19 nov. 2025
DEBUG 19 11. 2025
debug 15 jun. 2024
DEBUG 15 06. 2024
debug 15 mai. 2024
DEBUG 15 05. 2024
d